In [ ]:
import os
import sys
import json
import time
from pathlib import Path

from typing import List, Literal
from pydantic import BaseModel, Field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from util.text_similarity import rank_texts

from openai import OpenAI

from tqdm.auto import tqdm


In [ ]:
client = OpenAI()

In [ ]:
REPO_ROOT = Path.cwd().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

ARTIFACT_OUTPUT_DIR = REPO_ROOT / "results/protein_tokenization/artifacts"
annotation_counts_df = pd.read_csv(ARTIFACT_OUTPUT_DIR / "annotation_counts.csv")
display(annotation_counts_df.head())

In [ ]:
semantic_annotation_matches_df = pd.read_csv(ARTIFACT_OUTPUT_DIR / "semantic_annotation_matches.csv")
display(semantic_annotation_matches_df.head())

In [ ]:
developer_prompt = """
You are classifying biological annotations for antimicrobial resistance (AMR).

AMR includes:
- antibiotic resistance proteins
- efflux pumps specific to antibiotics
- beta-lactamases
- target modification/protection
- antimicrobial peptide resistance

NOT AMR:
- antibiotic biosynthesis
- metabolism
- virulence
- metal resistance
- generic transport unless clearly resistance-related

Examples:
Annotation: methicillin resistance factor FemA
Category: antibiotic_resistance
is_amr: true

Annotation: phenazine antibiotic biosynthesis protein
Category: non_amr_antibiotic_biosynthesis
is_amr: false

Annotation: antibiotic ABC transporter
Category: ambiguous
is_amr: false
"""


class AMRLabel(BaseModel):
    annotation: str
    is_amr: bool
    category: Literal[
        "antibiotic_resistance",
        "non_amr_antibiotic_biosynthesis",
        "non_amr_transport",
        "non_amr_metal_resistance",
        "non_amr_virulence",
        "ambiguous",
        "unknown",
    ]
    confidence: float = Field(ge=0.0, le=1.0)
    evidence_terms: List[str]
    reason: str


In [ ]:
batch_size = 5
output_csv = ARTIFACT_OUTPUT_DIR / "semantic_annotation_amr_labels.csv"

source_df = semantic_annotation_matches_df.reset_index().rename(columns={"index": "row_idx"})

if output_csv.exists():
    saved_results_df = pd.read_csv(output_csv)
    completed_row_idxs = set(saved_results_df["row_idx"].tolist())
else:
    saved_results_df = pd.DataFrame()
    completed_row_idxs = set()

pending_df = source_df[~source_df["row_idx"].isin(completed_row_idxs)].copy()
results_batches = []

for batch_start in tqdm(range(0, len(pending_df), batch_size)):
    batch_df = pending_df.iloc[batch_start : batch_start + batch_size]
    batch_results = []

    try:
        for _, row in batch_df.iterrows():
            annotation = row["annotation"]

            response = client.responses.parse(
                model="gpt-5.4-mini",
                input=[
                    {"role": "system", "content": developer_prompt},
                    {"role": "user", "content": f"Classify this annotation: {annotation}"},
                ],
                text_format=AMRLabel,
            )

            parsed = response.output_parsed.model_dump()
            parsed["row_idx"] = row["row_idx"]
            batch_results.append(parsed)

        batch_results_df = pd.DataFrame(batch_results)
        results_batches.append(batch_results_df)
        saved_results_df = pd.concat([saved_results_df, batch_results_df], ignore_index=True)
        saved_results_df.to_csv(output_csv, index=False)

        print(f"Saved batch {batch_start // batch_size + 1} ({len(batch_results_df)} rows) to {output_csv.name}")
    except Exception as exc:
        print(f"Stopped at batch {batch_start // batch_size + 1}. Previously saved results remain in {output_csv.name}. Error: {exc}")
        break

if output_csv.exists():
    results_df = pd.read_csv(output_csv)
else:
    results_df = pd.concat(results_batches, ignore_index=True) if results_batches else pd.DataFrame()

final_df = source_df.merge(
    results_df,
    on="row_idx",
    how="left",
    suffixes=("_original", "_pred"),
)

print(final_df)

In [ ]:
display(final_df)